In [ ]:
# ---- EXPERIMENT CONFIG: edit this cell for each run --------------------------
# Clifford Flow (GATr vector field, the reference recipe of pose3d/pose3d/config.py) on SYMSOL I with 10 % of the
# training data, with and without the STF second track (the "two-tower" model, pose3d/pose3d/models/stf_track.py).
# Protocol of the symsol1_10pct_*_i2s_adamw runs (reports/i2s_symsol1_10pct.md): 4,500 of 45,000 training views per
# shape (tet, cube, icosa, cone, cyl; 22,500 images), scored on the first 1,000 test views per shape (5,000
# images) by the rotation error to the NEAREST EQUIVALENT pose: acc@15 is the primary metric, the median error the
# auxiliary one (no flow log-likelihood: an ODE likelihood is not what the IPDF / Image2Sphere papers report); ResNet-50, no
# augmentation, no EMA, AdamW lr 1e-4 x sqrt(64/32), 5 warmup epochs + cosine, global batch 64, 40 epochs.
# The only difference between the variants below is the model flags, so a difference in the scores is the model's.
#
# RUN ONE VARIANT PER KERNEL (set RUN): gpu_pool/rtx_monitor.py keys its state by kernel and re-pushes the first
# run's points when a kernel runs two run names in sequence (reports/i2s_symsol1_10pct.md, infrastructure notes).
#   control        the reference model: Clifford MLP condition head + GATr vector field, nothing else.
#   stf_l3         + --stf_track --stf_lmax 3: STF tensors of rank 2, 3 (5 + 7 components per channel).
#   stf_l6         + --stf_track --stf_lmax 6: ranks 2..6 (5 + 7 + 9 + 11 + 13).
#   stf_l3_frame   (optional) stf_l3 with --stf_inject frame: the tensors in the body frame of the rotor; gives up
#                  the exact conjugation equivariance of the control (see the report) for absolute information.
#   frame_ch       the reference model with --pose_tokens frame_ch: the pose as the frame R e1, R e2, R e3, three vector channels of one
#                  GATr token (instead of one rotor token), the velocity read out in the camera frame; GATr's reflection / rotation
#                  action then moves the pose the way a camera rotation does (R -> G R) instead of conjugating it (R -> G R G^T).
#   control_nc, stf_l3_nc (optional)   the same with --symsol_nearest_coupling: each training start rotation flows
#                  to the nearest equivalent pose, which reports/symsol_p1_stall.md found necessary for the GATr
#                  flow to learn the polyhedra at all; the likelihood keeps its meaning. Run these if the plain
#                  control stalls on tet/cube/icosa (median error ~ 50-60 deg, log-likelihood ~ -2.29 on them).
# Estimated wall time per variant on one RTX Pro 6000: ~40 min (control), ~50 min (stf_l3), ~60 min (stf_l6); on a T4
# ~2.7 / 3.5 / 4.1 h (pose3d/reports/symsol_stf/README.md). Each is over the 30-minute limit of CLAUDE.md: a human
# approves the launch.
#
# OFFLINE (RTX Pro 6000 has no internet) attach: the repo snapshot dataset (a ga-research-*.tar.gz written by
# pose3d/scripts/make_snapshot.py, or an auto-extracted folder holding pose3d/pose3d/config.py),
#   syfry5suvzovvakmuj/ga-research-offline-deps   (wheels: clifford, e3nn, healpy, image2sphere, ...; ResNet weights)
#   syfry5suvzovvakmuj/ga-research-offline-deps-py313   (the healpy wheel for the Python 3.13 image)
#   syfry5suvzovvakmuj/ga-research-gatr-deps      (the GATr wheel and its pure-python helpers)
#   syfry5suvzovvakmuj/symsol1-10pct-fulltest     (symsol1_t4500_v5000/symsol/{train,test}/rotations.npz + images/)
# W&B then runs offline and gpu_pool/rtx_monitor.py replays the WANDB_SYNC lines into W&B. ONLINE (T4): the branch
# is cloned, the dependencies come from PyPI / GitHub and W&B syncs live from the `wandb_api_key` secret.
RUN = 'frame_ch'
WANDB_API_KEY = None
REPO = 'chagrygoris/GA-Research'
BRANCH = 'claude/bold-ritchie-r2t9p9'
# One process (--no-ddp): the loader keeps the decoded images in RAM (22,500 + 5,000 images = 4.1 GB) and a DDP rank
# would hold a copy each. --symsol_test_views 1000 scores the same first 1,000 test views per shape as the other
# symsol1_10pct_* runs. The speed options (dense GATr linears, torch.compile, bf16 backbone) are the defaults.
COMMON = ['--model', 'clifford_flow', '--encoder', 'resnet50', '--dataset', 'symsol', '--symsol_set', '1',
          '--symsol_train_views', '4500', '--symsol_test_views', '1000', '--no-use_warp', '--no-use_synth', '--no-ema',
          '--fixed_val_noise', '--no-ddp', '--batch_size', '64', '--n_epochs', '40', '--optimizer', 'adamw',
          '--lr', '1e-4', '--lr_scaling', 'sqrt', '--lr_schedule', 'cosine', '--warmup_epochs', '5', '--num_workers', '4',
          '--no-symsol_likelihood']   # acc@15 primary, median error auxiliary; no flow likelihood
VARIANTS = {
    'symsol1_10pct_gatr_control': [],
    'symsol1_10pct_gatr_stf_l3': ['--stf_track', '--stf_lmax', '3'],
    'symsol1_10pct_gatr_stf_l6': ['--stf_track', '--stf_lmax', '6'],
    'symsol1_10pct_gatr_stf_l3_frame': ['--stf_track', '--stf_lmax', '3', '--stf_inject', 'frame'],
    'symsol1_10pct_gatr_frame_ch': ['--pose_tokens', 'frame_ch'],
    'symsol1_10pct_gatr_control_nc': ['--symsol_nearest_coupling'],
    'symsol1_10pct_gatr_stf_l3_nc': ['--stf_track', '--stf_lmax', '3', '--symsol_nearest_coupling'],
}
SHORT = {'control': 'symsol1_10pct_gatr_control', 'frame_ch': 'symsol1_10pct_gatr_frame_ch', 'stf_l3': 'symsol1_10pct_gatr_stf_l3',
         'stf_l6': 'symsol1_10pct_gatr_stf_l6', 'stf_l3_frame': 'symsol1_10pct_gatr_stf_l3_frame',
         'control_nc': 'symsol1_10pct_gatr_control_nc', 'stf_l3_nc': 'symsol1_10pct_gatr_stf_l3_nc'}
assert RUN in SHORT, f'RUN must be one of {list(SHORT)}'
SNAPSHOT_MARKER = 'stf_tower'   # a repo snapshot / clone must carry the STF two-tower model (models/clifford_flow.py)
ARGS = COMMON


In [ ]:
import glob, os, shutil, socket, subprocess, tarfile

def online(host='github.com', timeout=5):
    try:
        socket.create_connection((host, 443), timeout=timeout).close()
        return True
    except OSError:
        return False

ONLINE = online()
print('internet:', ONLINE)
if os.path.exists('/kaggle/working/GA-Research'):
    shutil.rmtree('/kaggle/working/GA-Research')
os.chdir('/kaggle/working')
markers = glob.glob('/kaggle/input/**/pose3d/pose3d/config.py', recursive=True)
tgzs = glob.glob('/kaggle/input/**/ga-research-*.tar.gz', recursive=True)
if markers:        # a repo snapshot dataset, auto-extracted by Kaggle
    assert len(markers) == 1, f'attach exactly one repo snapshot, found {markers}'
    shutil.copytree(markers[0][:markers[0].index('/pose3d/pose3d/config.py')], '/kaggle/working/GA-Research')
elif tgzs:
    assert len(tgzs) == 1, f'attach exactly one repo snapshot, found {tgzs}'
    with tarfile.open(tgzs[0]) as t:
        t.extractall('/kaggle/working')
else:
    assert ONLINE, 'no internet and no repo snapshot attached'
    r = subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, f'https://github.com/{REPO}.git'],
                       capture_output=True, text=True)
    print(r.stdout + r.stderr); r.check_returncode()
print(subprocess.run(['git', '-C', 'GA-Research', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)
src = open('/kaggle/working/GA-Research/pose3d/pose3d/models/clifford_flow.py').read()
assert SNAPSHOT_MARKER in src, f'stale code: {SNAPSHOT_MARKER!r} is not in models/clifford_flow.py'


In [ ]:
# Dependencies Kaggle lacks. Online: PyPI / GitHub, with the image's torch pinned by a constraints file.
# Offline: the wheels of the ga-research-offline-deps and ga-research-gatr-deps datasets (installed without
# dependencies: the image's torch must stay).
import importlib, importlib.metadata as md, sys
# ga-research-offline-deps holds the wheels and ResNet weights; ga-research-offline-deps-py313 adds the healpy
# wheel for the Python 3.13 image (October 2026). Every ga-research-offline-deps* folder attached is read.
deps = sorted(glob.glob('/kaggle/input/**/ga-research-offline-deps*', recursive=True))
gatr_deps = glob.glob('/kaggle/input/**/ga-research-gatr-deps', recursive=True)
if deps:
    wheels = sorted(w for d in deps for w in glob.glob(os.path.join(d, '*.whl')))
    if gatr_deps:
        wheels += sorted(glob.glob(os.path.join(gatr_deps[0], '*.whl')))
    # Keep only the wheels this image's Python can run: the datasets may hold one healpy build per Python version
    # (the Kaggle image moved from 3.12 to 3.13 in October 2026), and pip rejects the whole command on one bad wheel.
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename
    supported = set(sys_tags())
    wheels = [w for w in wheels if parse_wheel_filename(os.path.basename(w))[3] & supported]
    print('wheels:', [os.path.basename(w) for w in wheels])
    assert any('healpy' in w for w in wheels), 'no healpy wheel for this Python: attach a ga-research-offline-deps* dataset that has one'
    subprocess.run(['pip', 'install', '-q', '--no-index', '--no-deps', '--disable-pip-version-check', *wheels], check=True)
    ckpt_dir = os.path.expanduser('~/.cache/torch/hub/checkpoints'); os.makedirs(ckpt_dir, exist_ok=True)
    for w in glob.glob('/kaggle/input/**/resnet*-*.pth', recursive=True):   # ImageNet weights (resnet50, resnet101)
        shutil.copy(w, os.path.join(ckpt_dir, os.path.basename(w)))
else:
    assert ONLINE, 'no internet and no ga-research-offline-deps attached'
    cons = []
    for p in ['torch', 'torchvision', 'numpy', 'scipy', 'pillow', 'tqdm', 'wandb', 'scikit-image', 'matplotlib',
              'pandas', 'sympy', 'astropy', 'packaging']:
        try:
            cons.append(f'{p}=={md.version(p)}')
        except md.PackageNotFoundError:
            pass
    open('/kaggle/working/constraints.txt', 'w').write('\n'.join(cons) + '\n')
    subprocess.run(['pip', 'install', '-q', '-c', '/kaggle/working/constraints.txt', 'e3nn==0.5.9', 'healpy==1.19.0',
                    'git+https://github.com/chagrygoris/image2sphere.git',
                    'git+https://github.com/chagrygoris/clifford-group-equivariant-neural-networks.git'], check=True)
    # GATr, the vector field: --no-deps, its pins would replace torch.
    subprocess.run(['pip', 'install', '-q', '--no-deps', 'einops', 'opt_einsum',
                    'git+https://github.com/Qualcomm-AI-research/geometric-algebra-transformer.git'], check=True)
for mod in ['e3nn', 'healpy', 'image2sphere', 'clifford']:
    importlib.import_module(mod)
# GATr hard-imports xformers (only for attention masks the flow never passes); the repo installs a stub before
# importing it (models/gatr_denoiser._import_gatr), so check the import the way the training process does.
sys.path.insert(0, '/kaggle/working/GA-Research/pose3d')
from pose3d.models.gatr_denoiser import _import_gatr
_import_gatr()
print('python', sys.version.split()[0], '| deps import cleanly')


In [ ]:
# W&B: live when online (the wandb_api_key secret, or WANDB_API_KEY above); offline files + WANDB_SYNC lines otherwise.
key = WANDB_API_KEY
if key is None and ONLINE:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('wandb_api_key')
    except Exception as e:   # no secret attached: fall back to offline logging
        print('wandb_api_key secret not available:', e)
if key and ONLINE:
    import wandb
    wandb.login(key=key)
    os.environ['WANDB_API_KEY'] = key
else:
    os.environ['WANDB_MODE'] = 'offline'
print('wandb mode:', os.environ.get('WANDB_MODE', 'online'))


In [ ]:
os.chdir('/kaggle/working/GA-Research/pose3d')   # the package is found from here, no install needed
found = glob.glob('/kaggle/input/**/symsol/test/rotations.npz', recursive=True)
assert found, 'attach the SYMSOL dataset (symsol/{train,test}/rotations.npz + images/)'
DATA = found[0][:-len('/symsol/test/rotations.npz')]
print('SYMSOL root:', DATA, flush=True)
name = SHORT[RUN]
cmd = [sys.executable, '-u', '-m', 'pose3d', f'--path_to_datasets={DATA}', '--platform=kaggle', f'--run_name={name}',
       *COMMON, *VARIANTS[name]]
print(' '.join(cmd), flush=True)
# The output is also written to /kaggle/working/<run_name>.log: Kaggle keeps /kaggle/working as the kernel's output
# even when its own log stream comes back empty (seen on long RTX sessions), so the per-epoch metrics survive.
# WATCHDOG: a run of this protocol takes ~30 min on an RTX Pro 6000 and ~3 h on a T4; the training process is
# killed after MAX_MINUTES so that a stuck session cannot burn the 12-hour GPU budget (two sessions did exactly that).
gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
MAX_MINUTES = 90 if 'RTX' in gpu_name else 330
print(f'GPU: {gpu_name!r}, watchdog {MAX_MINUTES} min', flush=True)
import threading, time
with open(f'/kaggle/working/{name}.log', 'w') as fh:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ})
    def _watchdog():
        time.sleep(MAX_MINUTES * 60)
        if proc.poll() is None:
            print(f'WATCHDOG: killing the run after {MAX_MINUTES} minutes', flush=True)
            proc.kill()
    threading.Thread(target=_watchdog, daemon=True).start()
    for line in proc.stdout:
        print(line, end='', flush=True)
        fh.write(line); fh.flush()
    rc = proc.wait()
# a crash marks the Kaggle run as failed instead of passing silently.
assert rc == 0, f'pose3d exited with code {rc}'
